In [1]:
# Создаёт ~/mlproject/tests/{__init__.py, conftest.py, test_predictor.py}
import os
from pathlib import Path

PROJECT = Path(os.path.expanduser("~/mlproject"))
TESTS = PROJECT / "tests"
TESTS.mkdir(parents=True, exist_ok=True)

INIT_CODE = ""

CONFTEST_CODE = r'''import os, sys
import joblib
import pytest

PROJECT = os.path.expanduser("~/mlproject")
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

import src.predictor as pred_mod
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


@pytest.fixture
def fake_artifacts(tmp_path, monkeypatch):
    """Маленькие фейковые .joblib + подмена _DEFAULT_ART."""
    spans = ["a@b.com", "c@d.com", "+7 900 111-22-33", "+7 900 444-55-66",
             "123-456-789 00", "987-654-321 00"]
    labels = ["EMAIL", "EMAIL", "PHONE_NUMBER", "PHONE_NUMBER", "SNILS", "SNILS"]

    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=1)
    X = vec.fit_transform(spans)
    clf = LogisticRegression(max_iter=1000, random_state=42).fit(X, labels)

    joblib.dump(vec, tmp_path / "lab2_span_tfidf.joblib")
    joblib.dump(clf, tmp_path / "lab2_span_logreg.joblib")

    monkeypatch.setattr(pred_mod, "_DEFAULT_ART", tmp_path)
    monkeypatch.setattr(pred_mod, "_VECTORIZER", None)
    monkeypatch.setattr(pred_mod, "_MODEL", None)
    monkeypatch.setattr(pred_mod, "_EXPECTED_N_FEATURES", None)
    yield tmp_path


class ModelSpy:
    def __init__(self):
        self.called = False
    def predict_proba(self, X):
        self.called = True
        raise AssertionError("model.predict_proba must NOT be called on invalid input")
    @property
    def classes_(self):
        return ["EMAIL", "PHONE_NUMBER", "SNILS"]
    n_features_in_ = None


@pytest.fixture
def model_spy(fake_artifacts, monkeypatch):
    pred_mod._ensure_model()
    spy = ModelSpy()
    spy.n_features_in_ = pred_mod._EXPECTED_N_FEATURES
    monkeypatch.setattr(pred_mod, "_MODEL", spy)
    yield spy
    assert not spy.called, "модель была вызвана при невалидном входе"
'''

TEST_CODE = r'''import os
import pytest

import src.predictor as pred_mod
from src.predictor import (
    predict, predict_from_file,
    PredictorError, ValidationError, InputTooLargeError, FormatError,
)


def test_positive_correct_input(fake_artifacts):
    result = predict(["a@b.com", "+7 900 111-22-33"])
    assert isinstance(result, list) and len(result) == 2
    for r in result:
        assert set(r) >= {"span", "label", "confidence"}
        assert isinstance(r["label"], str)
        assert 0.0 <= r["confidence"] <= 1.0


def test_negative_wrong_type(model_spy):
    with pytest.raises(ValidationError) as ei:
        predict(42)
    assert ei.value.code == "validation_error"
    assert not model_spy.called


def test_negative_none_and_empty(model_spy):
    with pytest.raises(ValidationError):
        predict(None)
    with pytest.raises(ValidationError):
        predict([])
    assert not model_spy.called


def test_negative_nan_inf(model_spy):
    with pytest.raises(ValidationError):
        predict([float("nan")])
    with pytest.raises(ValidationError):
        predict([float("inf")])
    assert not model_spy.called


def test_negative_too_many_spans(model_spy, monkeypatch):
    monkeypatch.setattr(pred_mod, "MAX_SPANS_PER_REQUEST", 3)
    with pytest.raises(InputTooLargeError) as ei:
        predict(["a", "b", "c", "d"])
    assert ei.value.code == "input_too_large"
    assert not model_spy.called


def test_negative_span_too_long(model_spy, monkeypatch):
    monkeypatch.setattr(pred_mod, "MAX_SPAN_LEN_CHARS", 10)
    with pytest.raises(InputTooLargeError):
        predict(["x" * 11])
    assert not model_spy.called


def test_negative_nul_and_ctrl(model_spy):
    with pytest.raises(ValidationError):
        predict(["ok", "\x00"])
    with pytest.raises(ValidationError):
        predict(["a\x01b"])
    assert not model_spy.called


def test_negative_file_too_large(model_spy, tmp_path, monkeypatch):
    monkeypatch.setattr(pred_mod, "MAX_FILE_BYTES", 100)
    p = tmp_path / "big.txt"
    p.write_bytes(b"a" * 200)
    with pytest.raises(InputTooLargeError) as ei:
        predict_from_file(p)
    assert ei.value.code == "input_too_large"
    assert not model_spy.called


def test_negative_file_binary_magic(model_spy, tmp_path):
    p = tmp_path / "fake.txt"
    p.write_bytes(b"\x89PNG\r\n\x1a\n" + b"\x00" * 64)
    with pytest.raises(FormatError) as ei:
        predict_from_file(p)
    assert ei.value.code == "format_error"
    assert not model_spy.called


def test_negative_file_bad_extension(model_spy, tmp_path):
    p = tmp_path / "data.csv"
    p.write_text("a,b,c\n", encoding="utf-8")
    with pytest.raises(FormatError):
        predict_from_file(p)
    assert not model_spy.called


def test_negative_file_bad_encoding(model_spy, tmp_path):
    p = tmp_path / "cp1251.txt"
    p.write_bytes("кириллица".encode("cp1251"))
    with pytest.raises(FormatError):
        predict_from_file(p)
    assert not model_spy.called


def test_negative_file_missing(model_spy, tmp_path):
    with pytest.raises(ValidationError):
        predict_from_file(tmp_path / "does_not_exist.txt")
    assert not model_spy.called
'''

(TESTS / "__init__.py").write_text(INIT_CODE, encoding="utf-8")
(TESTS / "conftest.py").write_text(CONFTEST_CODE, encoding="utf-8")
(TESTS / "test_predictor.py").write_text(TEST_CODE, encoding="utf-8")

print(f"OK: {TESTS}/__init__.py")
print(f"OK: {TESTS}/conftest.py ({len(CONFTEST_CODE)} байт)")
print(f"OK: {TESTS}/test_predictor.py ({len(TEST_CODE)} байт)")

# Проверка, что src/predictor.py на месте
pred_path = PROJECT / "src" / "predictor.py"
if not pred_path.exists():
    print("⚠️  src/predictor.py не найден — выполните ячейку 1 из lab2_predict.ipynb")
else:
    print(f"OK: src/predictor.py существует")

OK: /home/mluser/mlproject/tests/__init__.py
OK: /home/mluser/mlproject/tests/conftest.py (1806 байт)
OK: /home/mluser/mlproject/tests/test_predictor.py (3153 байт)
OK: src/predictor.py существует


In [2]:
import subprocess, sys

# pytest должен быть в текущем venv
r = subprocess.run([sys.executable, "-m", "pytest", "--version"],
                   capture_output=True, text=True)
print("pytest:", r.stdout.strip() or r.stderr.strip())

if r.returncode != 0:
    print("Устанавливаю pytest...")
    subprocess.run([sys.executable, "-m", "pip", "install", "pytest"], check=True)
    r = subprocess.run([sys.executable, "-m", "pytest", "--version"],
                       capture_output=True, text=True)
    print("pytest:", r.stdout.strip())

# Проверка sklearn (нужен conftest.py)
try:
    import sklearn
    print("sklearn:", sklearn.__version__)
except ImportError:
    print("Устанавливаю scikit-learn...")
    subprocess.run([sys.executable, "-m", "pip", "install", "scikit-learn"], check=True)

pytest: /home/mluser/venv/bin/python: No module named pytest
Устанавливаю pytest...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pytest]━━━━━━━━━━━ 2/3 [pytest]
pytest: pytest 9.2.0
sklearn: 1.9.1


In [3]:
import subprocess, sys, os

PROJECT = os.path.expanduser("~/mlproject")
TESTS = os.path.join(PROJECT, "tests")

# -v — подробно, чтобы видеть имя каждого теста и PASSED/FAILED
r = subprocess.run(
    [sys.executable, "-m", "pytest", TESTS, "-v", "--tb=short"],
    cwd=PROJECT,
    capture_output=True,
    text=True,
)

print(r.stdout)
if r.stderr.strip():
    print("--- STDERR ---")
    print(r.stderr[-3000:])

print("=" * 60)
print(f"Код возврата: {r.returncode} "
      f"({'все тесты пройдены' if r.returncode == 0 else 'есть падения'})")
print("=" * 60)

============================= test session starts ==============================
platform linux -- Python 3.11.2, pytest-9.2.0, pluggy-1.7.0 -- /home/mluser/venv/bin/python
cachedir: .pytest_cache
rootdir: /home/mluser/mlproject
plugins: anyio-4.15.1, platformdirs-4.13.6
collecting ... collected 12 items

tests/test_predictor.py::test_positive_correct_input PASSED              [  8%]
tests/test_predictor.py::test_negative_wrong_type PASSED                 [ 16%]
tests/test_predictor.py::test_negative_none_and_empty PASSED             [ 25%]
tests/test_predictor.py::test_negative_nan_inf PASSED                    [ 33%]
tests/test_predictor.py::test_negative_too_many_spans PASSED             [ 41%]
tests/test_predictor.py::test_negative_span_too_long PASSED              [ 50%]
tests/test_predictor.py::test_negative_nul_and_ctrl PASSED               [ 58%]
tests/test_predictor.py::test_negative_file_too_large PASSED             [ 66%]
tests/test_predictor.py::test_negative_file_binary_ma

In [7]:
import re, subprocess, sys, os, json

PROJECT = os.path.expanduser("~/mlproject")
TESTS = os.path.join(PROJECT, "tests")

# --color=no — ключевой флаг, иначе в stdout попадают ANSI-коды
r = subprocess.run(
    [sys.executable, "-m", "pytest", TESTS, "-v", "--tb=no", "--color=no"],
    cwd=PROJECT, capture_output=True, text=True,
)

print("=" * 70)
print("ПОЛНЫЙ ВЫВОД PYTEST")
print("=" * 70)
print(r.stdout)
if r.stderr.strip():
    print("--- STDERR ---")
    print(r.stderr[-2000:])

# Страховка: вырезаем ANSI на случай, если --color=no не сработал
ANSI_RE = re.compile(r"\x1b\[[0-9;]*m")

rows = []
for line in r.stdout.splitlines():
    line_clean = ANSI_RE.sub("", line)
    m = re.search(r"(tests/\S+\.py)::(\S+)\s+(PASSED|FAILED|ERROR|SKIPPED)", line_clean)
    if m:
        rows.append({"file": m.group(1), "test": m.group(2), "status": m.group(3)})

# Запасной парсер: summary, если имена всё равно не распарсились
if not rows:
    m_p = re.search(r"(\d+) passed", r.stdout)
    m_f = re.search(r"(\d+) failed", r.stdout)
    passed_total = int(m_p.group(1)) if m_p else 0
    failed_total = int(m_f.group(1)) if m_f else 0
    if passed_total or failed_total:
        print(f"\n⚠️  Имена не распарсились, но summary: "
              f"passed={passed_total}, failed={failed_total}")

print("\n" + "=" * 70)
print("ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("=" * 70)
print(f"{'#':<3} {'Тест':<45} {'Статус'}")
print("-" * 70)
for i, row in enumerate(rows, 1):
    print(f"{i:<3} {row['test']:<45} {row['status']}")

passed = sum(1 for x in rows if x["status"] == "PASSED")
failed = sum(1 for x in rows if x["status"] != "PASSED")
print("-" * 70)
print(f"Всего: {len(rows)} | PASSED: {passed} | FAILED: {failed}")

summary_path = os.path.join(PROJECT, "metrics", "test_results.json")
os.makedirs(os.path.dirname(summary_path), exist_ok=True)
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump({"rows": rows, "passed": passed, "failed": failed,
               "pytest_returncode": r.returncode}, f, indent=2, ensure_ascii=False)
print(f"\nСохранено: {summary_path}")

ПОЛНЫЙ ВЫВОД PYTEST
============================= test session starts ==============================
platform linux -- Python 3.11.2, pytest-9.2.0, pluggy-1.7.0 -- /home/mluser/venv/bin/python
cachedir: .pytest_cache
rootdir: /home/mluser/mlproject
plugins: anyio-4.15.1, platformdirs-4.13.6
collecting ... collected 12 items

tests/test_predictor.py::test_positive_correct_input PASSED              [  8%]
tests/test_predictor.py::test_negative_wrong_type PASSED                 [ 16%]
tests/test_predictor.py::test_negative_none_and_empty PASSED             [ 25%]
tests/test_predictor.py::test_negative_nan_inf PASSED                    [ 33%]
tests/test_predictor.py::test_negative_too_many_spans PASSED             [ 41%]
tests/test_predictor.py::test_negative_span_too_long PASSED              [ 50%]
tests/test_predictor.py::test_negative_nul_and_ctrl PASSED               [ 58%]
tests/test_predictor.py::test_negative_file_too_large PASSED             [ 66%]
tests/test_predictor.py::test_neg